# Detecting Metastatic Cancer in Histopathology Patches — Full Dataset
**Deep Learning PBL · Saif Khattab (202312556) · Al-Zaytoonah University of Jordan**

Full run on all **220,025 labelled patches**, with the fixes identified in the 50k prototype:

| Issue found in prototype | Fix in this version |
|---|---|
| Validation AUC oscillated; EarlyStopping may have stopped too early | Lower baseline LR (5e-4), EarlyStopping patience 7 + `min_delta`, LR is reduced (patience 2) several times before stopping, more max epochs |
| EfficientNetB0 under-performed (96 px input → 3×3 feature map; very conservative fine-tuning) | Upscale to 128 px inside the model, fine-tune the **whole** backbone (BatchNorm frozen) with LR 1e-4 |
| Grad-CAM maps empty for confident predictions (sigmoid saturation) | Gradients taken w.r.t. the **logit** |
| Possible slide-level data leakage in a random split | Internal split kept for full analysis **plus** predictions on Kaggle's hidden test set (slide-disjoint PCam split) → `submission.csv` → Kaggle AUC; the gap between the two AUCs measures leakage |
| Speed on full data | Mixed precision (float16) + index-based `tf.data` pipeline (no copy of the 6 GB image array) |
| Robustness of final predictions | Test-time augmentation (8 rotations / flips) used identically for validation, internal test and Kaggle test |

> **Before running:** Settings → Accelerator → **GPU T4 x2**, Settings → **Internet on**. Expected run time ≈ 2–2.5 h.

## 1. Setup

In [ ]:
import os, glob, time, random, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Configuration ----------------
SEED          = 42
SUBSET_SIZE   = None        # None -> all 220,025 labelled images (set e.g. 50_000 for a quick test)
IMG_SIZE      = 96
TL_SIZE       = 128         # EfficientNet input size (upscaled inside the model)
BATCH_SIZE    = 128
# Baseline CNN
BASE_LR       = 5e-4
EPOCHS_BASE   = 30          # upper bound; EarlyStopping decides
# Transfer learning
EPOCHS_HEAD   = 3           # stage 1: frozen backbone
EPOCHS_FT     = 15          # stage 2: fine-tuning (upper bound)
HEAD_LR       = 1e-3
FT_LR         = 1e-4
# Early stopping / LR schedule (fix for unstable validation curve)
ES_PATIENCE   = 7
ES_MIN_DELTA  = 1e-4
LR_PATIENCE   = 2
# Evaluation
TARGET_RECALL = 0.95        # clinical operating point (sensitivity)
USE_TTA       = True
OUT_DIR       = '/kaggle/working'

keras.mixed_precision.set_global_policy('mixed_float16')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
set_seed(SEED)

print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPUs      :', tf.config.list_physical_devices('GPU'))
print('Precision :', keras.mixed_precision.global_policy().name)

## 2. Locate the data & EDA

In [ ]:
candidates = ['/kaggle/input/histopathologic-cancer-detection',
              '/kaggle/input/competitions/histopathologic-cancer-detection']
DATA_DIR = next((d for d in candidates if os.path.exists(os.path.join(d, 'train_labels.csv'))), None)
if DATA_DIR is None:
    found = glob.glob('/kaggle/input/*/train_labels.csv') + glob.glob('/kaggle/input/*/*/train_labels.csv')
    assert found, 'train_labels.csv not found - check that the competition data is attached (Input panel).'
    DATA_DIR = os.path.dirname(found[0])
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

df = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
kaggle_test_ids = sorted(f[:-4] for f in os.listdir(TEST_DIR) if f.endswith('.tif'))
print('Data dir                 :', DATA_DIR)
print('Labelled training images :', len(df))
print('Kaggle test images       :', len(kaggle_test_ids), '(labels hidden)')
print('Missing values           :', int(df.isna().sum().sum()))
print('Duplicate ids            :', int(df['id'].duplicated().sum()))
df.head()

In [ ]:
counts = df['label'].value_counts().sort_index()
print(pd.DataFrame({'count': counts, 'percent': (counts / len(df) * 100).round(2)}))

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['0 - No tumour', '1 - Tumour'], counts.values, color=['#4C72B0', '#DD8452'])
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}\n({v/len(df):.1%})', ha='center', va='bottom')
ax.set_title('Class distribution (all labelled images)')
ax.set_ylabel('Number of patches'); ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.show()

## 3. Stratified 70 / 15 / 15 split

In [ ]:
if SUBSET_SIZE:
    df_sub, _ = train_test_split(df, train_size=SUBSET_SIZE, stratify=df['label'], random_state=SEED)
else:
    df_sub = df.copy()
df_sub = df_sub.reset_index(drop=True)

train_df, temp_df = train_test_split(df_sub, test_size=0.30, stratify=df_sub['label'], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

split_summary = pd.DataFrame({
    name: {'images': len(d), 'positives': int(d['label'].sum()), 'positive %': round(d['label'].mean() * 100, 2)}
    for name, d in [('train', train_df), ('val', val_df), ('internal test', test_df)]
}).T
print(split_summary)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(f'{OUT_DIR}/split_{name}.csv', index=False)

## 4. Load images into memory (uint8, ≈7.7 GB in total)

In [ ]:
def load_image(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('RGB'), dtype=np.uint8)

def load_images(ids, folder, workers=32):
    t0 = time.time()
    paths = [os.path.join(folder, f'{i}.tif') for i in ids]
    out = np.empty((len(paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    def work(k):
        out[k] = load_image(paths[k])
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(work, range(len(paths))))
    print(f'  loaded {len(paths):,} images in {time.time() - t0:.0f}s')
    return out

print('Train       :'); X_train = load_images(train_df['id'].values, TRAIN_DIR)
print('Validation  :'); X_val   = load_images(val_df['id'].values,   TRAIN_DIR)
print('Int. test   :'); X_test  = load_images(test_df['id'].values,  TRAIN_DIR)
print('Kaggle test :'); X_kag   = load_images(kaggle_test_ids,       TEST_DIR)
y_train = train_df['label'].values.astype('float32')
y_val   = val_df['label'].values.astype('float32')
y_test  = test_df['label'].values.astype('float32')
total_gb = sum(a.nbytes for a in (X_train, X_val, X_test, X_kag)) / 1e9
print(f'RAM used by images: {total_gb:.2f} GB')

### 4.1 Sample patches (green box = central 32×32 region that defines the label)

In [ ]:
def draw_center(ax):
    ax.add_patch(Rectangle((31.5, 31.5), 32, 32, fill=False, edgecolor='lime', linewidth=1.5))

def show_samples(X, y, n=8, seed=SEED):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4.8))
    for row, label in enumerate([0, 1]):
        idx = rng.choice(np.where(y == label)[0], n, replace=False)
        for ax, i in zip(axes[row], idx):
            ax.imshow(X[i]); draw_center(ax); ax.axis('off')
        axes[row, 0].set_title('Label 0 - no tumour' if label == 0 else 'Label 1 - tumour', loc='left')
    plt.tight_layout(); plt.show()

show_samples(X_train, y_train)

### 4.2 Quality check: blank / background-only / dark patches

In [ ]:
def image_stats(X, chunk=4000):
    means, stds = [], []
    for i in range(0, len(X), chunk):
        c = X[i:i + chunk].reshape(len(X[i:i + chunk]), -1).astype(np.float32)
        means.append(c.mean(1)); stds.append(c.std(1))
    return np.concatenate(means), np.concatenate(stds)

def flag_patches(X):
    m, s = image_stats(X)
    return (m > 230) | (m < 25) | (s < 5), m, s

tr_flag, tr_mean, tr_std = flag_patches(X_train)
print(f'Flagged (very bright / very dark / nearly uniform): {tr_flag.sum()} ({tr_flag.mean():.2%} of train)')
print('Labels of flagged images:', pd.Series(y_train[tr_flag]).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, color in [(0, '#4C72B0'), (1, '#DD8452')]:
    axes[0].hist(tr_mean[y_train == label], bins=60, alpha=0.6, label=f'label {label}', color=color)
    axes[1].hist(tr_std[y_train == label],  bins=60, alpha=0.6, label=f'label {label}', color=color)
axes[0].set_title('Mean pixel intensity per patch'); axes[1].set_title('Pixel std per patch')
for a in axes: a.legend()
plt.tight_layout(); plt.show()

pos_flag = np.where(tr_flag & (y_train == 1))[0][:8]
if len(pos_flag):
    fig, axes = plt.subplots(1, len(pos_flag), figsize=(2 * len(pos_flag), 2.4), squeeze=False)
    for ax, i in zip(axes[0], pos_flag):
        ax.imshow(X_train[i]); draw_center(ax); ax.set_title(f'm={tr_mean[i]:.0f}'); ax.axis('off')
    plt.suptitle('Flagged patches labelled TUMOUR (possible label noise)'); plt.tight_layout(); plt.show()
# Decision: flagged patches are kept (they also occur at inference time); their error rate is analysed at the end.

## 5. `tf.data` pipeline (index-based) + augmentation
Batches are gathered from the in-memory arrays by index, so the 6 GB image array is never copied into the graph.
Augmentation (training only) covers all 8 rotations/reflections — they keep the central 32×32 region in place — plus
per-image brightness/contrast and per-batch saturation/hue jitter to simulate stain variation. **No cropping.**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment_batch(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    b = tf.shape(x)[0]
    # all 8 dihedral transforms: independent LR flip, UD flip and transpose per image
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [2]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [1]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.transpose(x, [0, 2, 1, 3]), x)
    # per-image brightness and contrast
    x = x + tf.random.uniform([b, 1, 1, 1], -0.10, 0.10)
    mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - mean) * tf.random.uniform([b, 1, 1, 1], 0.90, 1.10) + mean
    x = tf.clip_by_value(x, 0.0, 1.0)
    # per-batch saturation and hue (stain variation)
    x = tf.image.adjust_saturation(x, tf.random.uniform([], 0.90, 1.10))
    x = tf.image.adjust_hue(x, tf.random.uniform([], -0.04, 0.04))
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x * 255.0, y

def to_float(x, y):
    return tf.cast(x, tf.float32), y

def make_ds(X, y=None, training=False, batch=BATCH_SIZE):
    if y is None:
        y = np.zeros(len(X), dtype='float32')
    idx = tf.data.Dataset.range(len(X))
    if training:
        idx = idx.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    idx = idx.batch(batch)
    def gather(i):
        xb, yb = tf.numpy_function(lambda j: (X[j], y[j]), [i], [tf.uint8, tf.float32])
        xb.set_shape([None, IMG_SIZE, IMG_SIZE, 3]); yb.set_shape([None])
        return xb, yb
    ds = idx.map(gather, num_parallel_calls=AUTOTUNE)
    ds = ds.map(augment_batch if training else to_float, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val)
print('Steps per epoch:', int(np.ceil(len(X_train) / BATCH_SIZE)))

# Visual check of augmentation
aug, _ = augment_batch(tf.constant(np.repeat(X_train[:1], 8, axis=0)), tf.zeros(8))
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, im in zip(axes, aug.numpy().astype('uint8')):
    ax.imshow(im); draw_center(ax); ax.axis('off')
plt.suptitle('Augmentations of one training patch'); plt.tight_layout(); plt.show()

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train.astype(int))
CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])}
print('Class weights:', CLASS_WEIGHT)

def get_metrics():
    return [keras.metrics.AUC(name='auc'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.BinaryAccuracy(name='accuracy')]

def get_callbacks(weights_path, patience=ES_PATIENCE):
    return [
        keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=patience,
                                      min_delta=ES_MIN_DELTA, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=LR_PATIENCE,
                                          min_delta=ES_MIN_DELTA, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(weights_path, monitor='val_auc', mode='max',
                                        save_best_only=True, save_weights_only=True),
    ]

def plot_history(histories, title):
    hist = {}
    for h in histories:
        for k, v in h.history.items():
            hist.setdefault(k, []).extend(v)
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    for ax, m in zip(axes[:2], ['loss', 'auc']):
        ax.plot(hist[m], label='train'); ax.plot(hist[f'val_{m}'], label='validation')
        ax.set_title(f'{title} - {m}'); ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
    lr_key = 'learning_rate' if 'learning_rate' in hist else ('lr' if 'lr' in hist else None)
    if lr_key:
        axes[2].plot(hist[lr_key]); axes[2].set_yscale('log'); axes[2].set_title('learning rate'); axes[2].grid(alpha=0.3)
    if len(histories) > 1:
        for a in axes: a.axvline(len(histories[0].history['loss']) - 0.5, color='gray', ls='--', lw=1)
    plt.tight_layout(); plt.show()

## 5b. Model, evaluation and Grad-CAM helpers

In [ ]:
def conv_block(x, filters, dropout, pool=True, last_name=None):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=last_name)(x)
    x = layers.BatchNormalization()(x)
    if pool:
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout)(x)
    return x

def build_baseline():
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inp)
    x = conv_block(x, 32,  0.10)                                     # 48x48x32
    x = conv_block(x, 64,  0.20)                                     # 24x24x64
    x = conv_block(x, 128, 0.30)                                     # 12x12x128
    x = conv_block(x, 256, 0.0, pool=False, last_name='last_conv')   # 12x12x256
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
    return keras.Model(inp, out, name='baseline_cnn')

def build_transfer():
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Resizing(TL_SIZE, TL_SIZE, interpolation='bilinear', name='upscale')(inp)
    try:
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)
    except Exception as e:
        raise RuntimeError('Could not download ImageNet weights. In the Kaggle editor open '
                           'Settings -> Turn on internet, then re-run this cell.') from e
    y = layers.GlobalAveragePooling2D(name='gap')(base.output)
    y = layers.Dropout(0.3)(y)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(y)
    return keras.Model(inp, out, name='efficientnetb0_tl'), base

def predict(model, X):
    base = make_ds(X, batch=512)
    if not USE_TTA:
        return model.predict(base.map(lambda x, y: x), verbose=0).ravel()
    preds = []
    for t in range(8):
        def tf_t(x, y, t=t):
            if t & 1: x = tf.reverse(x, [2])
            if t & 2: x = tf.reverse(x, [1])
            if t & 4: x = tf.transpose(x, [0, 2, 1, 3])
            return x
        preds.append(model.predict(base.map(tf_t), verbose=0).ravel())
    return np.mean(preds, axis=0)

def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1   = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return {'threshold': round(float(thr), 4), 'AUC': roc_auc_score(y, p), 'Accuracy': (tp + tn) / len(y),
            'Sensitivity (Recall)': sens, 'Specificity': spec, 'Precision': prec, 'F1': f1,
            'FN (missed tumours)': int(fn), 'FP (false alarms)': int(fp)}

def threshold_for_recall(y, p, target):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr >= target)])


def make_gradcam(model, layer_name):
    # Gradients are taken w.r.t. the LOGIT (pre-sigmoid score): for confident predictions the sigmoid
    # saturates (p ~ 1.0) and its gradient vanishes, which would give empty heat-maps.
    out_layer  = model.get_layer('output')          # final Dense(1, sigmoid)
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, out_layer.input])
    def gradcam(img):
        x = tf.convert_to_tensor(img[None].astype('float32'))
        with tf.GradientTape() as tape:
            conv_out, feats = grad_model(x, training=False)
            logit = tf.matmul(tf.cast(feats, tf.float32), tf.cast(out_layer.kernel, tf.float32)) \
                    + tf.cast(out_layer.bias, tf.float32)
            score = logit[:, 0]
        # gradient w.r.t. the tensor actually on the path to the score; cast afterwards (mixed precision)
        grads = tf.cast(tape.gradient(score, conv_out), tf.float32)
        conv_out = tf.cast(conv_out, tf.float32)
        weights = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
        cam = cam / (tf.reduce_max(cam) + 1e-8)
        cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE), method='bilinear')[..., 0].numpy()
        return cam, float(tf.sigmoid(logit)[0, 0])
    return gradcam

def centre_ratio(cam):
    return float(cam[32:64, 32:64].sum() / (cam.sum() + 1e-8))

## 5c. Preflight check (≈2 min)
Runs every later step on 512 images (both models, both training stages, TTA prediction, Grad-CAM, submission writing)
so that any error appears **now**, not after 2 hours of training.

In [ ]:
t0 = time.time()
Xs, ys = X_train[:512], y_train[:512]
small = make_ds(Xs, ys, training=True, batch=64)
for name, builder, layer in [('baseline', build_baseline, 'last_conv'), ('efficientnet', None, 'top_activation')]:
    if builder is None:
        m, b = build_transfer()
        for l in b.layers: l.trainable = False
        m.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss='binary_crossentropy', metrics=get_metrics())
        m.fit(small, epochs=1, steps_per_epoch=2, class_weight=CLASS_WEIGHT, verbose=0)
        for l in b.layers:
            if not isinstance(l, layers.BatchNormalization): l.trainable = True
    else:
        m = builder()
    m.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=get_metrics())
    m.fit(small, validation_data=make_ds(Xs[:128], ys[:128]), epochs=1, steps_per_epoch=2,
          class_weight=CLASS_WEIGHT, callbacks=get_callbacks('/tmp/pf.weights.h5'), verbose=0)
    m.load_weights('/tmp/pf.weights.h5')
    p = predict(m, Xs[:64]); assert p.shape == (64,) and np.isfinite(p).all()
    evaluate(ys[:64], p, threshold_for_recall(ys[:64], p, TARGET_RECALL))
    cam, _ = make_gradcam(m, layer)(Xs[0]); assert cam.shape == (IMG_SIZE, IMG_SIZE) and np.isfinite(cam).all()
    print(f'  {name}: OK (cam max={cam.max():.2f})')
pd.DataFrame({'id': kaggle_test_ids[:64], 'label': predict(m, X_kag[:64])}).to_csv('/tmp/sub_test.csv', index=False)
del m, small; keras.backend.clear_session(); gc.collect(); set_seed(SEED)
print(f'Preflight passed in {time.time() - t0:.0f}s')

## 6. Model 1 — Baseline CNN (from scratch)

In [ ]:
set_seed(SEED)
baseline = build_baseline()
baseline.compile(optimizer=keras.optimizers.Adam(BASE_LR), loss='binary_crossentropy', metrics=get_metrics())
baseline.summary()

In [ ]:
BASE_W = f'{OUT_DIR}/baseline.weights.h5'
t0 = time.time()
h_base = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_BASE,
                      class_weight=CLASS_WEIGHT, callbacks=get_callbacks(BASE_W), verbose=2)
baseline.load_weights(BASE_W)
best_ep = int(np.argmax(h_base.history['val_auc'])) + 1
print(f'Baseline: {len(h_base.epoch)} epochs run, best epoch = {best_ep}, '
      f'best val AUC = {max(h_base.history["val_auc"]):.4f}, time = {(time.time() - t0) / 60:.1f} min')
plot_history([h_base], 'Baseline CNN')

## 7. Model 2 — Transfer learning with EfficientNetB0 (128 px input)

In [ ]:
tl_model, tl_base = build_transfer()
assert tl_model.get_layer('top_activation') is not None

# ---- Stage 1: train only the new head ----
for layer in tl_base.layers:
    layer.trainable = False
tl_model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss='binary_crossentropy', metrics=get_metrics())
print('Trainable params (stage 1):', int(sum(np.prod(w.shape) for w in tl_model.trainable_weights)))

HEAD_W = f'{OUT_DIR}/tl_head.weights.h5'
t0 = time.time()
h_head = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                      class_weight=CLASS_WEIGHT, callbacks=get_callbacks(HEAD_W, patience=EPOCHS_HEAD), verbose=2)
tl_model.load_weights(HEAD_W)
val_auc_head = max(h_head.history['val_auc'])
print(f'Stage 1 done in {(time.time() - t0) / 60:.1f} min - best val AUC = {val_auc_head:.4f}')

In [ ]:
# ---- Stage 2: fine-tune the whole backbone (BatchNorm layers stay frozen) ----
for layer in tl_base.layers:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
tl_model.compile(optimizer=keras.optimizers.Adam(FT_LR), loss='binary_crossentropy', metrics=get_metrics())
print('Trainable params (stage 2):', int(sum(np.prod(w.shape) for w in tl_model.trainable_weights)))

FT_W = f'{OUT_DIR}/tl_finetune.weights.h5'
t0 = time.time()
h_ft = tl_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                    class_weight=CLASS_WEIGHT, callbacks=get_callbacks(FT_W), verbose=2)
tl_model.load_weights(FT_W)
val_auc_ft = max(h_ft.history['val_auc'])
print(f'Stage 2 done in {(time.time() - t0) / 60:.1f} min - best val AUC = {val_auc_ft:.4f}')

if val_auc_ft < val_auc_head:
    print('Fine-tuning did not improve validation AUC; reverting to stage-1 weights.')
    tl_model.load_weights(HEAD_W)

plot_history([h_head, h_ft], 'EfficientNetB0 (dashed line = start of fine-tuning)')

## 8. Evaluation
* Predictions use **test-time augmentation** (mean over the 8 rotations/reflections) when `USE_TTA=True`,
  applied identically to validation, internal test and Kaggle test.
* Model selection and the clinical threshold use the **validation set only**; the internal test set is used **once**.
* Clinical note: a False Negative (missed tumour) is more dangerous than a False Positive, so besides 0.5 we report
  the threshold that reaches `TARGET_RECALL` sensitivity on validation.

In [ ]:
t0 = time.time()
models = {'Baseline CNN': baseline, 'EfficientNetB0': tl_model}
p_val = {name: predict(m, X_val) for name, m in models.items()}
print(f'Validation predictions ({"with" if USE_TTA else "without"} TTA) in {(time.time() - t0) / 60:.1f} min')
val_table = pd.DataFrame({m: evaluate(y_val, p, 0.5) for m, p in p_val.items()}).T
print('Validation results @ threshold 0.5'); display(val_table.round(4))

plt.figure(figsize=(5, 4.5))
for m, p in p_val.items():
    fpr, tpr, _ = roc_curve(y_val, p)
    plt.plot(fpr, tpr, label=f'{m} (AUC={roc_auc_score(y_val, p):.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate (1 - Specificity)'); plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC curves - validation set'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

BEST_NAME  = max(p_val, key=lambda m: roc_auc_score(y_val, p_val[m]))
best_model = models[BEST_NAME]
GRADCAM_LAYER = 'top_activation' if BEST_NAME == 'EfficientNetB0' else 'last_conv'
CLIN_THR = threshold_for_recall(y_val, p_val[BEST_NAME], TARGET_RECALL)
print(f'Selected model: {BEST_NAME}')
print(f'Threshold for >= {TARGET_RECALL:.0%} sensitivity on validation: {CLIN_THR:.4f}')

In [ ]:
# ---- Final, one-time evaluation on the internal held-out test set ----
p_test = predict(best_model, X_test)
test_table = pd.DataFrame({'default (0.5)': evaluate(y_test, p_test, 0.5),
                           f'clinical (recall>={TARGET_RECALL:.0%} on val)': evaluate(y_test, p_test, CLIN_THR)}).T
print(f'INTERNAL TEST results - {BEST_NAME}'); display(test_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, thr) in zip(axes, [('threshold 0.5', 0.5), (f'clinical threshold {CLIN_THR:.3f}', CLIN_THR)]):
    cm = confusion_matrix(y_test, (p_test >= thr).astype(int), labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    ax.set_title(f'Internal test confusion matrix\n{name}')
plt.tight_layout(); plt.show()

## 9. Kaggle hidden test set → `submission.csv` (leakage check)
Kaggle's test images come from the PCam validation/test slides, which are **different slides** from the training data.
Submitting these predictions returns an AUC that is free of slide-level leakage. Compare it with the internal test AUC above:
a clearly lower Kaggle AUC means the random internal split was optimistic.

In [ ]:
t0 = time.time()
p_kag = predict(best_model, X_kag)
sub = pd.DataFrame({'id': kaggle_test_ids, 'label': p_kag})
sub.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'submission.csv written: {sub.shape[0]:,} rows in {(time.time() - t0) / 60:.1f} min')
print('Predicted positive rate @0.5 - Kaggle test: {:.1%} | internal test: {:.1%}'.format(
      (p_kag >= 0.5).mean(), (p_test >= 0.5).mean()))
sub.head()

## 10. Grad-CAM explainability & error analysis
Gradients are taken w.r.t. the **logit** (pre-sigmoid score) so confident predictions still give meaningful maps.
*Centre ratio* = share of Grad-CAM mass inside the central 32×32 box (uniform map = 1/9 ≈ 0.11).

In [ ]:
gradcam = make_gradcam(best_model, GRADCAM_LAYER)

pred_test = (p_test >= CLIN_THR).astype(int)
def pick(mask, key, n=6):
    idx = np.where(mask)[0]
    return idx[np.argsort(key[idx])][:n]
groups = {
    'True positives (most confident)':  pick((y_test == 1) & (pred_test == 1), -p_test),
    'False negatives (missed tumours)': pick((y_test == 1) & (pred_test == 0),  p_test),
    'False positives (false alarms)':   pick((y_test == 0) & (pred_test == 1), -p_test),
}
for title, idx in groups.items():
    if len(idx) == 0:
        print(f'{title}: none'); continue
    fig, axes = plt.subplots(2, len(idx), figsize=(2.3 * len(idx), 5), squeeze=False)
    for j, i in enumerate(idx):
        cam, p = gradcam(X_test[i])
        axes[0, j].imshow(X_test[i]); draw_center(axes[0, j]); axes[0, j].set_title(f'p={p_test[i]:.3f}'); axes[0, j].axis('off')
        axes[1, j].imshow(X_test[i]); axes[1, j].imshow(cam, cmap='jet', alpha=0.45); draw_center(axes[1, j])
        axes[1, j].set_title(f'centre={centre_ratio(cam):.2f}'); axes[1, j].axis('off')
    plt.suptitle(f'{title} - {BEST_NAME} (threshold {CLIN_THR:.3f})'); plt.tight_layout(); plt.show()

tp_idx = np.where((y_test == 1) & (pred_test == 1))[0][:300]
ratios = [centre_ratio(gradcam(X_test[i])[0]) for i in tp_idx]
print(f'Mean Grad-CAM centre ratio on {len(tp_idx)} true positives: {np.mean(ratios):.3f} (uniform = 0.111)')

te_flag, _, _ = flag_patches(X_test)
err = pred_test != y_test
if te_flag.sum():
    print(f'Error rate on flagged test patches : {err[te_flag].mean():.2%} (n={te_flag.sum()})')
print(f'Error rate on other test patches   : {err[~te_flag].mean():.2%}')

## 10b. Ensemble (mean of both models)
Model choice stays on the **validation set**; the ensemble is one more candidate compared there.

In [ ]:
# ---- Ensemble (mean of the two models' TTA probabilities) ----
t0 = time.time()
other = 'EfficientNetB0' if BEST_NAME == 'Baseline CNN' else 'Baseline CNN'
p_val_ens = (p_val['Baseline CNN'] + p_val['EfficientNetB0']) / 2
val_cmp = pd.DataFrame({m: evaluate(y_val, p, 0.5) for m, p in {**p_val, 'Ensemble (mean)': p_val_ens}.items()}).T
print('VALIDATION comparison @0.5 (model choice is made here, on validation only)'); display(val_cmp.round(4))

p_test_other = predict(models[other], X_test)
p_test_by = {BEST_NAME: p_test, other: p_test_other}
p_test_ens = (p_test_by['Baseline CNN'] + p_test_by['EfficientNetB0']) / 2
thr_ens = threshold_for_recall(y_val, p_val_ens, TARGET_RECALL)
ens_table = pd.DataFrame({'Ensemble @0.5': evaluate(y_test, p_test_ens, 0.5),
                          f'Ensemble clinical (recall>={TARGET_RECALL:.0%} on val)': evaluate(y_test, p_test_ens, thr_ens),
                          f'{other} @0.5': evaluate(y_test, p_test_other, 0.5)}).T
print('INTERNAL TEST - ensemble and second model'); display(ens_table.round(4))

p_kag_other = predict(models[other], X_kag)
p_kag_by = {BEST_NAME: p_kag, other: p_kag_other}
p_kag_ens = (p_kag_by['Baseline CNN'] + p_kag_by['EfficientNetB0']) / 2
pd.DataFrame({'id': kaggle_test_ids, 'label': p_kag_ens}).to_csv(f'{OUT_DIR}/submission_ensemble.csv', index=False)
pd.DataFrame({'id': kaggle_test_ids, 'label': p_kag_other}).to_csv(f'{OUT_DIR}/submission_{other.split()[0].lower()}.csv', index=False)
pd.DataFrame({'id': test_df['id'].values, 'label': y_test.astype(int), 'p_baseline': p_test_by['Baseline CNN'],
              'p_efficientnet': p_test_by['EfficientNetB0'], 'p_ensemble': p_test_ens}).to_csv(f'{OUT_DIR}/test_predictions_all.csv', index=False)
val_cmp.to_csv(f'{OUT_DIR}/val_results_with_ensemble.csv'); ens_table.to_csv(f'{OUT_DIR}/test_results_ensemble.csv')
print(f'Ensemble threshold: {thr_ens:.4f} | done in {(time.time() - t0) / 60:.1f} min')
print(sorted(os.listdir(OUT_DIR)))

## 11. Save artefacts

In [ ]:
pd.DataFrame({'id': val_df['id'].values, 'label': y_val.astype(int),
              **{f'p_{m}': p for m, p in p_val.items()}}).to_csv(f'{OUT_DIR}/val_predictions.csv', index=False)
pd.DataFrame({'id': test_df['id'].values, 'label': y_test.astype(int),
              f'p_{BEST_NAME}': p_test}).to_csv(f'{OUT_DIR}/test_predictions.csv', index=False)
val_table.to_csv(f'{OUT_DIR}/val_results.csv'); test_table.to_csv(f'{OUT_DIR}/test_results.csv')
best_model.save(f'{OUT_DIR}/best_model.keras')
with open(f'{OUT_DIR}/run_config.json', 'w') as f:
    json.dump({'subset_size': SUBSET_SIZE, 'n_train': len(X_train), 'n_val': len(X_val), 'n_test': len(X_test),
               'seed': SEED, 'best_model': BEST_NAME, 'clinical_threshold': CLIN_THR,
               'target_recall': TARGET_RECALL, 'tta': USE_TTA, 'mixed_precision': True,
               'baseline_lr': BASE_LR, 'ft_lr': FT_LR, 'tl_size': TL_SIZE}, f, indent=2)
print(sorted(os.listdir(OUT_DIR)))

## 12. Summary & limitations
- **Leakage:** the internal split is random at patch level (no slide IDs in the Kaggle data), so internal-test scores
  may be optimistic. The Kaggle hidden-test AUC (slide-disjoint) is the leakage-free estimate; the clinical threshold is
  also tuned on the (patch-level) validation set and may need re-calibration on new slides.
- **Stain variation:** handled with colour augmentation only; stain normalisation (e.g. Macenko) is possible future work.
- **Label noise:** some near-blank patches carry a tumour label (see 4.2).
- The system is an **assistive tool only** — final diagnosis remains with a qualified pathologist.

In [ ]:
print(f'Best model: {BEST_NAME} | TTA: {USE_TTA}')
display(val_table.round(4))
display(test_table.round(4))
print('Kaggle AUC: submit /kaggle/working/submission.csv and compare with the internal test AUC above.')